# Trinity AI — Benchmark Analysis

**Author:** Derek J. Russell  
**System:** Trinity AI · JARVIS + J-Prime + Reactor-Core  
**Hardware:** GCP g2-standard-4 · NVIDIA L4 (23 GB VRAM)  
**Model:** Qwen2.5-Coder-14B-Instruct · Q4_K_M quantization  

---

This notebook loads every recorded demo run from `history.json` and visualises:

1. **Inference throughput** — tok/s per task across all runs  
2. **Latency trends** — end-to-end response time per task  
3. **Token volume** — completion tokens generated per run  
4. **Governance test reliability** — pass rate and throughput over time  
5. **System summary** — headline numbers for the Palantir Fellowship pitch  

Each new demo run appends to `history.json` automatically — re-run this notebook at any time to include fresh data.

In [ ]:
import os
import json
import math
from pathlib import Path

# Fix matplotlib cache dir — avoids warnings when ~/.matplotlib is not writable
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl_trinity_cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "axes.titlecolor":  "#e6edf3",
    "axes.titlesize":   13,
    "axes.titleweight": "bold",
    "axes.grid":        True,
    "grid.color":       "#21262d",
    "grid.linewidth":   0.8,
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#c9d1d9",
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "legend.fontsize":  9,
    "font.family":      "monospace",
    "figure.dpi":       120,
})

CYAN   = "#58a6ff"
GREEN  = "#3fb950"
RED    = "#f85149"
YELLOW = "#d29922"
PURPLE = "#bc8cff"
DIM    = "#8b949e"

BENCH_DIR = Path(".")   # run from the benchmarks/ directory
# If running from the repo root:
if not (BENCH_DIR / "history.json").exists():
    BENCH_DIR = Path("benchmarks")

print(f"Benchmark directory: {BENCH_DIR.resolve()}")

In [ ]:
# ── Load history ─────────────────────────────────────────────────────────────

history_path = BENCH_DIR / "history.json"
with history_path.open() as f:
    raw_history = json.load(f)

print(f"Loaded {len(raw_history)} run(s) from history.json")

# ── Flatten into a tidy DataFrame ────────────────────────────────────────────

rows = []
for entry in raw_history:
    run_ts = entry.get("run_ts", "unknown")
    # Normalise timestamp to a sortable string (dashes → colons for display)
    run_label = run_ts.replace("T", " ").replace("-", ":", 2)

    sys_info = entry.get("system", {})

    for task_key in ("inference_0", "inference_1"):
        t = entry.get(task_key)
        if not t:
            continue
        rows.append({
            "run_ts":           run_ts,
            "run_label":        run_label,
            "task":             t["label"],
            "latency_ms":       t["latency_ms"],
            "latency_s":        t["latency_ms"] / 1000,
            "tok_s":            t["tok_s"],
            "completion_tokens": t["completion_tokens"],
            "model":            t.get("model", "jarvis-prime"),
            "artifact":         sys_info.get("artifact", ""),
            "context_window":   sys_info.get("context_window", 0),
        })

    tests = entry.get("tests")
    if tests:
        rows_test = {
            "run_ts":            run_ts,
            "run_label":         run_label,
            "tests_passed":      tests["passed"],
            "tests_failed":      tests["failed"],
            "pass_rate":         tests["pass_rate"],
            "duration_s":        tests["duration_s"],
            "tests_per_second":  tests["tests_per_second"],
        }
    else:
        rows_test = None

df = pd.DataFrame(rows)
df["run_idx"] = pd.Categorical(df["run_ts"]).codes

# Separate test rows into their own frame
test_rows = []
for entry in raw_history:
    t = entry.get("tests")
    if t:
        t["run_ts"]    = entry["run_ts"]
        t["run_label"] = entry["run_ts"].replace("T", " ").replace("-", ":", 2)
        test_rows.append(t)
df_tests = pd.DataFrame(test_rows) if test_rows else pd.DataFrame()

print(f"\nInference rows:  {len(df)}")
print(f"Test-suite rows: {len(df_tests)}")
df.head()

## 1 · Inference Throughput (tok/s)

In [ ]:
tasks = df["task"].unique()
runs  = df["run_ts"].unique()
n_runs  = len(runs)
n_tasks = len(tasks)

x       = np.arange(n_runs)
width   = 0.35
colors  = [CYAN, GREEN]

fig, ax = plt.subplots(figsize=(max(7, n_runs * 2.5), 5))

for i, (task, color) in enumerate(zip(tasks, colors)):
    subset = df[df["task"] == task].sort_values("run_ts")
    vals   = subset["tok_s"].values
    offset = (i - (n_tasks - 1) / 2) * width
    bars   = ax.bar(x + offset, vals, width, label=task, color=color, alpha=0.85, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{v:.1f}",
            ha="center", va="bottom", fontsize=8, color="#e6edf3",
        )

run_labels = [r.replace("T", "\n").replace("-", ":", 2) for r in runs]
ax.set_xticks(x)
ax.set_xticklabels(run_labels, fontsize=8)
ax.set_ylabel("Throughput (tok/s)")
ax.set_title("Inference Throughput per Run — NVIDIA L4 · Q4_K_M")
ax.legend(loc="upper left")
ax.set_ylim(0, max(df["tok_s"]) * 1.25)

# Reference line: L4 baseline expectation
ax.axhline(20, color=YELLOW, linewidth=0.8, linestyle="--", label="20 tok/s baseline", zorder=2)
ax.text(n_runs - 0.5, 20.4, "baseline 20 tok/s", fontsize=7, color=YELLOW)

fig.tight_layout()
plt.savefig(BENCH_DIR / "chart_throughput.png", bbox_inches="tight")
plt.show()
print("Saved → chart_throughput.png")

## 2 · End-to-End Latency

In [ ]:
fig, ax = plt.subplots(figsize=(max(7, n_runs * 2.5), 5))

for i, (task, color) in enumerate(zip(tasks, colors)):
    subset = df[df["task"] == task].sort_values("run_ts")
    vals   = subset["latency_s"].values
    offset = (i - (n_tasks - 1) / 2) * width
    bars   = ax.bar(x + offset, vals, width, label=task, color=color, alpha=0.85, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            f"{v:.1f}s",
            ha="center", va="bottom", fontsize=8, color="#e6edf3",
        )

ax.set_xticks(x)
ax.set_xticklabels(run_labels, fontsize=8)
ax.set_ylabel("Latency (seconds)")
ax.set_title("End-to-End Inference Latency per Run")
ax.legend(loc="upper left")
ax.set_ylim(0, max(df["latency_s"]) * 1.25)

fig.tight_layout()
plt.savefig(BENCH_DIR / "chart_latency.png", bbox_inches="tight")
plt.show()
print("Saved → chart_latency.png")

## 3 · Tokens Generated per Run

The jump from **250 → 631 tokens** on the infrastructure task reflects removing the
`max_tokens` cap so J-Prime writes the complete function instead of being cut off mid-generation.

In [ ]:
fig, ax = plt.subplots(figsize=(max(7, n_runs * 2.5), 5))

for i, (task, color) in enumerate(zip(tasks, colors)):
    subset = df[df["task"] == task].sort_values("run_ts")
    vals   = subset["completion_tokens"].values
    offset = (i - (n_tasks - 1) / 2) * width
    bars   = ax.bar(x + offset, vals, width, label=task, color=color, alpha=0.85, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2,
            str(int(v)),
            ha="center", va="bottom", fontsize=8, color="#e6edf3",
        )

ax.set_xticks(x)
ax.set_xticklabels(run_labels, fontsize=8)
ax.set_ylabel("Completion Tokens")
ax.set_title("Tokens Generated per Run (completion only, excludes prompt)")
ax.legend(loc="upper left")
ax.set_ylim(0, max(df["completion_tokens"]) * 1.3)

fig.tight_layout()
plt.savefig(BENCH_DIR / "chart_tokens.png", bbox_inches="tight")
plt.show()
print("Saved → chart_tokens.png")

## 4 · Throughput Consistency

Scatter plot of latency vs tokens generated — a perfectly linear relationship means
constant throughput regardless of generation length, which is the expected behaviour for
autoregressive decoding on a dedicated GPU.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for task, color in zip(tasks, colors):
    subset = df[df["task"] == task]
    ax.scatter(
        subset["completion_tokens"],
        subset["latency_s"],
        label=task, color=color, s=80, zorder=4, edgecolors="#0d1117", linewidths=0.5,
    )
    for _, row in subset.iterrows():
        ax.annotate(
            f"{row['tok_s']:.1f} tok/s",
            (row["completion_tokens"], row["latency_s"]),
            textcoords="offset points", xytext=(6, 4),
            fontsize=7, color=DIM,
        )

# Fit + plot a trend line across all points
if len(df) >= 2:
    xs = df["completion_tokens"].values
    ys = df["latency_s"].values
    coeffs = np.polyfit(xs, ys, 1)
    x_line = np.linspace(xs.min() * 0.8, xs.max() * 1.1, 100)
    ax.plot(
        x_line, np.polyval(coeffs, x_line),
        color=YELLOW, linewidth=1, linestyle="--", alpha=0.6, label="linear fit",
    )
    mean_tps = 1 / coeffs[0] if coeffs[0] != 0 else 0
    ax.text(
        0.98, 0.05,
        f"Implied throughput: {mean_tps:.1f} tok/s",
        transform=ax.transAxes, ha="right", fontsize=8, color=YELLOW,
    )

ax.set_xlabel("Completion Tokens")
ax.set_ylabel("Latency (s)")
ax.set_title("Latency vs Tokens — Linear = Consistent Throughput")
ax.legend()

fig.tight_layout()
plt.savefig(BENCH_DIR / "chart_consistency.png", bbox_inches="tight")
plt.show()
print("Saved → chart_consistency.png")

## 5 · Governance Test Reliability

In [ ]:
if df_tests.empty:
    print("No test data recorded yet — run the demo without --no-tests to capture it.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    run_labels_t = df_tests["run_label"].tolist()
    x_t = np.arange(len(df_tests))

    # ── Pass rate ───────────────────────────────────────────────────────────
    ax = axes[0]
    bars = ax.bar(x_t, df_tests["pass_rate"], color=GREEN, alpha=0.85, zorder=3)
    for bar, v in zip(bars, df_tests["pass_rate"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            f"{v:.1f}%",
            ha="center", va="bottom", fontsize=8, color="#e6edf3",
        )
    ax.set_xticks(x_t)
    ax.set_xticklabels(run_labels_t, fontsize=7, rotation=15)
    ax.set_ylabel("Pass Rate (%)")
    ax.set_ylim(95, 101)
    ax.set_title("Governance Test Pass Rate")
    ax.axhline(99, color=YELLOW, linewidth=0.8, linestyle="--")
    ax.text(len(df_tests) - 0.5, 99.1, "99% target", fontsize=7, color=YELLOW)

    # ── Tests passed ────────────────────────────────────────────────────────
    ax = axes[1]
    ax.bar(x_t, df_tests["tests_passed"], color=CYAN, alpha=0.85, zorder=3, label="passed")
    ax.bar(x_t, df_tests["tests_failed"], bottom=df_tests["tests_passed"],
           color=RED, alpha=0.6, zorder=3, label="failed (pre-existing)")
    for xi, (p, f) in enumerate(zip(df_tests["tests_passed"], df_tests["tests_failed"])):
        ax.text(xi, p + f + 5, f"{p:,}", ha="center", fontsize=8, color=GREEN)
    ax.set_xticks(x_t)
    ax.set_xticklabels(run_labels_t, fontsize=7, rotation=15)
    ax.set_ylabel("Test Count")
    ax.set_title("Tests Passed vs Pre-existing Failures")
    ax.legend(loc="lower right", fontsize=7)

    # ── Tests per second ────────────────────────────────────────────────────
    ax = axes[2]
    bars = ax.bar(x_t, df_tests["tests_per_second"], color=PURPLE, alpha=0.85, zorder=3)
    for bar, v in zip(bars, df_tests["tests_per_second"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{v:.0f}/s",
            ha="center", va="bottom", fontsize=8, color="#e6edf3",
        )
    ax.set_xticks(x_t)
    ax.set_xticklabels(run_labels_t, fontsize=7, rotation=15)
    ax.set_ylabel("Tests / Second")
    ax.set_title("Test Suite Execution Speed")

    fig.suptitle(
        "Ouroboros Governance Test Suite — Reliability Across Runs",
        fontsize=14, fontweight="bold", color="#e6edf3", y=1.02,
    )
    fig.tight_layout()
    plt.savefig(BENCH_DIR / "chart_governance_tests.png", bbox_inches="tight")
    plt.show()
    print("Saved → chart_governance_tests.png")

## 6 · Dashboard — All Metrics in One View

In [ ]:
has_tests = not df_tests.empty
n_rows = 2 if has_tests else 1
fig = plt.figure(figsize=(16, 5 * n_rows))
fig.patch.set_facecolor("#0d1117")

# ── Row 1: Throughput · Latency · Tokens ─────────────────────────────────────
ax1 = fig.add_subplot(n_rows, 3, 1)
ax2 = fig.add_subplot(n_rows, 3, 2)
ax3 = fig.add_subplot(n_rows, 3, 3)

def _grouped_bars(ax, metric, ylabel, fmt_fn):
    for i, (task, color) in enumerate(zip(tasks, colors)):
        subset = df[df["task"] == task].sort_values("run_ts")
        vals = subset[metric].values
        offset = (i - (n_tasks - 1) / 2) * width
        bars = ax.bar(x + offset, vals, width, label=task, color=color, alpha=0.85, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                fmt_fn(v),
                ha="center", va="bottom", fontsize=7, color="#e6edf3",
            )
    ax.set_xticks(x)
    ax.set_xticklabels(run_labels, fontsize=6)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.legend(fontsize=6)
    ax.set_ylim(0, df[metric].max() * 1.35)

_grouped_bars(ax1, "tok_s",            "tok/s",   lambda v: f"{v:.1f}")
_grouped_bars(ax2, "latency_s",        "s",       lambda v: f"{v:.1f}s")
_grouped_bars(ax3, "completion_tokens","tokens",  lambda v: str(int(v)))

ax1.set_title("Throughput",      fontsize=10)
ax2.set_title("Latency",         fontsize=10)
ax3.set_title("Tokens Generated",fontsize=10)

# ── Row 2: Governance tests (if data exists) ──────────────────────────────────
if has_tests:
    ax4 = fig.add_subplot(n_rows, 3, 4)
    ax5 = fig.add_subplot(n_rows, 3, 5)
    ax6 = fig.add_subplot(n_rows, 3, 6)

    x_t = np.arange(len(df_tests))
    rl_t = df_tests["run_label"].tolist()

    ax4.bar(x_t, df_tests["pass_rate"], color=GREEN, alpha=0.85, zorder=3)
    ax4.set_ylim(95, 101)
    ax4.set_xticks(x_t); ax4.set_xticklabels(rl_t, fontsize=6, rotation=15)
    ax4.set_ylabel("%", fontsize=8)
    ax4.set_title("Pass Rate", fontsize=10)
    ax4.axhline(99, color=YELLOW, linewidth=0.8, linestyle="--")

    ax5.bar(x_t, df_tests["tests_passed"], color=CYAN, alpha=0.85, zorder=3)
    ax5.set_xticks(x_t); ax5.set_xticklabels(rl_t, fontsize=6, rotation=15)
    ax5.set_ylabel("count", fontsize=8)
    ax5.set_title("Tests Passed", fontsize=10)

    ax6.bar(x_t, df_tests["tests_per_second"], color=PURPLE, alpha=0.85, zorder=3)
    ax6.set_xticks(x_t); ax6.set_xticklabels(rl_t, fontsize=6, rotation=15)
    ax6.set_ylabel("tests/s", fontsize=8)
    ax6.set_title("Suite Speed", fontsize=10)

fig.suptitle(
    "Trinity AI — Full Benchmark Dashboard",
    fontsize=15, fontweight="bold", color="#e6edf3", y=1.01,
)
fig.tight_layout()
plt.savefig(BENCH_DIR / "chart_dashboard.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved → chart_dashboard.png")

## 7 · Summary Statistics

In [ ]:
print("=" * 60)
print("  TRINITY AI — BENCHMARK SUMMARY")
print("=" * 60)
print(f"  Runs recorded:          {len(raw_history)}")
print()

for task in tasks:
    sub = df[df["task"] == task]
    print(f"  [{task}]")
    print(f"    Throughput:  {sub['tok_s'].mean():.1f} tok/s avg  "
          f"(range {sub['tok_s'].min():.1f}–{sub['tok_s'].max():.1f})")
    print(f"    Latency:     {sub['latency_s'].mean():.1f}s avg  "
          f"(range {sub['latency_s'].min():.1f}–{sub['latency_s'].max():.1f}s)")
    print(f"    Tokens gen:  {sub['completion_tokens'].mean():.0f} avg  "
          f"(range {sub['completion_tokens'].min():.0f}–{sub['completion_tokens'].max():.0f})")
    print()

if not df_tests.empty:
    print("  [Governance Tests]")
    print(f"    Pass rate:   {df_tests['pass_rate'].mean():.1f}% avg  "
          f"(range {df_tests['pass_rate'].min():.1f}–{df_tests['pass_rate'].max():.1f}%)")
    print(f"    Tests run:   {df_tests['tests_passed'].max():,} (latest run)")
    print(f"    Speed:       {df_tests['tests_per_second'].mean():.0f} tests/s avg")
    print()

print("  [Hardware]")
print("    GPU:   NVIDIA L4 (g2-standard-4 · 23 GB VRAM)")
print("    Model: Qwen2.5-Coder-14B-Instruct · Q4_K_M")
print("    Ctx:   8,192 tokens")
print("=" * 60)

## 8 · Palantir Fellowship Context

These benchmarks map directly to the four Fellowship evaluation criteria:

| Criterion | Evidence | Data Source |
|-----------|----------|-------------|
| **Live Cloud Inference** | ~24 tok/s on NVIDIA L4, all runs | `inference_0`, `inference_1` |
| **Pre-Execution Governance** | Every inference call is wrapped in risk classification + security gate | Demo Phase 3 video |
| **Verifiable Throughput** | Constant tok/s across different token lengths (see Chart 4) | Linear latency/tokens relationship |
| **Kernel Resilience** | >2,100 governance tests · >99% pass rate across runs | `tests` keys in history |

### Key Numbers for the Slide Deck

| Metric | Value |
|--------|-------|
| GPU inference speed | **~24 tok/s** (Q4_K_M · NVIDIA L4) |
| Infrastructure code generation | **631 tokens in 25.7s** (full function, ungated) |
| Threat analysis response | **133 tokens in 5.5s** |
| Governance tests | **2,132 passed · 99.3% pass rate** |
| Pre-existing failures | **14** (structural test harness issues, not pipeline bugs) |
| Developer team size | **1** |
| External funding | **$0** |

---
*Run `python3 demo_trinity_governed_loop.py --no-voice` to record a new run, then re-execute this notebook to include the latest data.*